# Layer Normalization


Use this notebook after the layer normalization note. The goal is to make normalization a concrete per-token operation rather than a black-box layer.

You will:
- match a manual LayerNorm calculation against PyTorch
- inspect zero mean and unit variance per token vector
- compare Pre-LN, Post-LN, and RMSNorm behavior


In [1]:
from pathlib import Path
import torch
import torch.nn as nn

LECTURE_NOTE_TITLE = 'Layer Normalization'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')

def manual_layer_norm(x, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps)

def rms_norm(x, eps=1e-5):
    return x / torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)


Lecture note: Layer Normalization


## Manual LayerNorm matches PyTorch LayerNorm


In [2]:
torch.manual_seed(0)
x = torch.randn(2, 4)
ln = nn.LayerNorm(4, elementwise_affine=False)
print(torch.allclose(manual_layer_norm(x), ln(x), atol=1e-6))


True


## LayerNorm zero-centers and rescales each token


In [3]:
y = manual_layer_norm(x)
print('mean:', y.mean(dim=-1))
print('std :', y.std(dim=-1, unbiased=False))


mean: tensor([2.2352e-08, 2.9802e-08])
std : tensor([1.0000, 1.0000])


## Epsilon on nearly constant inputs


In [4]:
near_constant = torch.tensor([[5.0, 5.0, 5.0, 5.0001]])
print('eps=1e-5:', manual_layer_norm(near_constant, eps=1e-5))
print('eps=1e-1:', manual_layer_norm(near_constant, eps=1e-1))


eps=1e-5: tensor([[-0.0078, -0.0078, -0.0078,  0.0238]])
eps=1e-1: tensor([[-7.8410e-05, -7.8410e-05, -7.8410e-05,  2.3825e-04]])


## Pre-LN and Post-LN residual paths


In [5]:
class PreLN(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln = nn.LayerNorm(4)
        self.linear = nn.Linear(4, 4)
    def forward(self, x):
        return x + self.linear(self.ln(x))

class PostLN(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln = nn.LayerNorm(4)
        self.linear = nn.Linear(4, 4)
    def forward(self, x):
        return self.ln(x + self.linear(x))

x = torch.randn(2, 4, requires_grad=True)
pre = PreLN(); post = PostLN()
pre(x).sum().backward(retain_graph=True)
pre_grad = x.grad.detach().clone(); x.grad.zero_()
post(x).sum().backward()
post_grad = x.grad.detach().clone()
print('Pre-LN grad norm :', pre_grad.norm().item())
print('Post-LN grad norm:', post_grad.norm().item())


Pre-LN grad norm : 3.9090709686279297
Post-LN grad norm: 2.0910570697196817e-07


## RMSNorm without mean subtraction


In [6]:
print('LayerNorm:', manual_layer_norm(x)[0])
print('RMSNorm  :', rms_norm(x)[0])


LayerNorm: tensor([-0.9685, -0.0551, -0.6140,  1.6376], grad_fn=<SelectBackward0>)
RMSNorm  : tensor([-1.3912, -0.7802, -1.1540,  0.3521], grad_fn=<SelectBackward0>)


## External reference repos


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/01_main-chapter-code/gpt.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py
